In [ ]:
!pip install medmnist
!python -m medmnist available
!pip install torchlogix


MedMNIST v3.0.2 @ https://github.com/MedMNIST/MedMNIST/
All available datasets:
	pathmnist       | PathMNIST       | Size: 28 (default), 64, 128, 224.
	chestmnist      | ChestMNIST      | Size: 28 (default), 64, 128, 224.
	dermamnist      | DermaMNIST      | Size: 28 (default), 64, 128, 224.
	octmnist        | OCTMNIST        | Size: 28 (default), 64, 128, 224.
	pneumoniamnist  | PneumoniaMNIST  | Size: 28 (default), 64, 128, 224.
	retinamnist     | RetinaMNIST     | Size: 28 (default), 64, 128, 224.
	breastmnist     | BreastMNIST     | Size: 28 (default), 64, 128, 224.
	bloodmnist      | BloodMNIST      | Size: 28 (default), 64, 128, 224.
	tissuemnist     | TissueMNIST     | Size: 28 (default), 64, 128, 224.
	organamnist     | OrganAMNIST     | Size: 28 (default), 64, 128, 224.
	organcmnist     | OrganCMNIST     | Size: 28 (default), 64, 128, 224.
	organsmnist     | OrganSMNIST     | Size: 28 (default), 64, 128, 224.
	organmnist3d    | OrganMNIST3D    | Size: 28 (default), 64.
	nodule

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
ROOT = "/content/drive/MyDrive/Colab Notebooks/ConfLGN/MOC"
os.chdir(ROOT)

print("Current working directory:", os.getcwd())
!pip install -e ../torchlogix

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current working directory: /content/drive/MyDrive/Colab Notebooks/ConfLGN/MOC
Obtaining file:///content/drive/MyDrive/Colab%20Notebooks/ConfLGN/torchlogix
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for torchlogix (pyproject.toml) ... done
  Created wheel for torchlogix: filename=torchlogix-0.1.1-0.editable-py3-none-any.whl size=5305 sha256=b078b320b9705f5d5747ecea65223a35f618ddf8a63c68ea78e8bd12e198316b
  Stored in directory: /tmp/pip-ephem-wheel-cache-d9btu_7s/wheels/cf/bf/dc/f5864c8cfbec5b2f942fd736ede23991967b0d80cbd239c9e3
Successfully built torchlogix
  Attempting uninstall: torchlogix
    Found existing installation: torchlogix 0.1.1
    Uninstalling torchlogix-0.1.1:
   

In [ ]:
  from pathlib import Path
  import sys

  from torchvision import transforms
  from medmnist import INFO, BloodMNIST
  import numpy as np

  PROJECT_ROOT = next(
      p for p in [Path.cwd(), *Path.cwd().parents]
      if (p / "MOC" / "utilities").exists()
  )

  if str(PROJECT_ROOT) not in sys.path:
      sys.path.insert(0, str(PROJECT_ROOT))

  from MOC.utilities.LogicNet import LogicNet
  from MOC.utilities.train_model import train_model
  transform = transforms.Compose([
      transforms.Grayscale(num_output_channels=1),  # LogicNet currently expects 1 channel
                     # LogicNet currently expects 28x28
      transforms.ToTensor(),
  ])

  target_transform = lambda y: int(np.asarray(y).item())
  num_classes = len(INFO["tissuemnist"]["label"])
  print(num_classes)
  train_dataset = BloodMNIST(
      split="train",
      transform=transform,
      target_transform=target_transform,
      download=True,
      size=28
  )
  test_dataset = BloodMNIST(
      split="test",
      transform=transform,
      target_transform=target_transform,
      download=True,
      size=28
  )


8


100%|██████████| 35.5M/35.5M [00:43<00:00, 824kB/s]


In [ ]:
import matplotlib.pyplot as plt

idx = 129  # numer obrazka z test setu

img, label = test_dataset[idx]

plt.imshow(img.squeeze())
plt.title(f"Label: {label}")
plt.axis("off")
plt.show()

In [ ]:
model = LogicNet(num_classes=num_classes,dense_num=4,base_dense_dims=[2048,1280,640,640], conv_num=3, kernel_multiplier=3, k=128,tau=20,channels=1)


In [15]:
model, history = train_model(
        model,
        train_dataset,
        test_dataset,
        lr=2e-1,
        weight_decay=0,
        batch_size=256,
        num_iterations=5000,
        metrics_every=250,
        force_cpu=False,
    )

cuda
iter  250 | train_loss 0.4767 | test_acc_discrete 0.8249 | test_loss_discrete 0.5032 | test_acc_relaxed 0.8100 | test_loss_relaxed 0.5258
iter  500 | train_loss 0.4693 | test_acc_discrete 0.8331 | test_loss_discrete 0.4856 | test_acc_relaxed 0.8372 | test_loss_relaxed 0.4860
iter  750 | train_loss 0.4512 | test_acc_discrete 0.8372 | test_loss_discrete 0.4871 | test_acc_relaxed 0.8351 | test_loss_relaxed 0.4847
iter 1000 | train_loss 0.4396 | test_acc_discrete 0.8354 | test_loss_discrete 0.4764 | test_acc_relaxed 0.8372 | test_loss_relaxed 0.4793


KeyboardInterrupt: 

In [19]:
import torch
torch.save(model.state_dict(), 'logicnet_bloodmnist.pth')

In [8]:
!ls


experiments  logicnet_bloodmnist.pth  utilities


In [18]:
# Interactive 3D chart for test_loss_acc. Run this after the training cell.
import numpy as np

try:
    import plotly.graph_objects as go
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'Plotly is required for this interactive 3D chart. '
        'Run `%pip install plotly` in a notebook cell, restart the kernel, '
        'then run this cell again.'
    ) from exc

if 'test_loss_acc' not in globals():
    raise NameError('Run the training cell first so test_loss_acc exists.')


def _as_1d_float_array(values):
    try:
        array = np.asarray(values, dtype=float).reshape(-1)
    except (TypeError, ValueError):
        return None

    if not np.all(np.isfinite(array)):
        return None

    return array.tolist()


if not isinstance(test_loss_acc, dict):
    raise TypeError('test_loss_acc must be a dictionary of metric lists.')

metric_series = {}
for metric_name, metric_values in test_loss_acc.items():
    numeric_values = _as_1d_float_array(metric_values)
    if numeric_values is not None:
        metric_series[metric_name] = numeric_values

if not metric_series:
    raise ValueError('No numeric metric lists were found in test_loss_acc.')

lengths = {metric_name: len(values) for metric_name, values in metric_series.items()}
if len(set(lengths.values())) != 1:
    raise ValueError(f'All plottable metrics must have the same length. Lengths: {lengths}')

point_count = next(iter(lengths.values()))
if point_count == 0:
    raise ValueError('test_loss_acc has no points yet. Run training long enough to record metrics.')

metric_names = list(metric_series.keys())


def _default_metric(preferred_name, fallback_index):
    if preferred_name in metric_names:
        return preferred_name
    return metric_names[min(fallback_index, len(metric_names) - 1)]


x_default = _default_metric('train_iteration', 0)
y_default = _default_metric('test_loss_discrete', 1)
z_default = _default_metric('test_acc_discrete', 2)

hover_text = [
    '<br>'.join(
        [f'point: {point_index}']
        + [
            f'{metric_name}: {metric_series[metric_name][point_index]:.6g}'
            for metric_name in metric_names
        ]
    )
    for point_index in range(point_count)
]

fig = go.Figure(
    data=[
        go.Scatter3d(
            x=metric_series[x_default],
            y=metric_series[y_default],
            z=metric_series[z_default],
            mode='markers+lines',
            marker=dict(
                size=6,
                color=list(range(point_count)),
                colorscale='Viridis',
                showscale=True,
                colorbar=dict(title='point'),
            ),
            line=dict(width=4, color='rgba(80, 80, 80, 0.45)'),
            text=hover_text,
            hovertemplate='%{text}<extra></extra>',
        )
    ]
)


def _axis_buttons(axis_name):
    axis_title_key = f'scene.{axis_name}axis.title.text'
    return [
        dict(
            label=metric_name,
            method='update',
            args=[
                {axis_name: [metric_series[metric_name]]},
                {axis_title_key: metric_name},
            ],
        )
        for metric_name in metric_names
    ]


fig.update_layout(
    title='test_loss_acc 3D metrics',
    template='plotly_white',
    height=650,
    margin=dict(l=0, r=0, t=120, b=0),
    scene=dict(
        xaxis_title=x_default,
        yaxis_title=y_default,
        zaxis_title=z_default,
    ),
    updatemenus=[
        dict(
            buttons=_axis_buttons('x'),
            active=metric_names.index(x_default),
            direction='down',
            x=0.00,
            y=1.16,
            xanchor='left',
            yanchor='top',
        ),
        dict(
            buttons=_axis_buttons('y'),
            active=metric_names.index(y_default),
            direction='down',
            x=0.24,
            y=1.16,
            xanchor='left',
            yanchor='top',
        ),
        dict(
            buttons=_axis_buttons('z'),
            active=metric_names.index(z_default),
            direction='down',
            x=0.48,
            y=1.16,
            xanchor='left',
            yanchor='top',
        ),
    ],
    annotations=[
        dict(text='X axis', x=0.00, y=1.24, xref='paper', yref='paper', showarrow=False),
        dict(text='Y axis', x=0.24, y=1.24, xref='paper', yref='paper', showarrow=False),
        dict(text='Z axis', x=0.48, y=1.24, xref='paper', yref='paper', showarrow=False),
    ],
)

fig.show()


NameError: name 'test_loss_acc' is not defined